# MENTORA — Train M1: Knowledge Gap Classifier (DeBERTa-v3-base)

Target: **F1 (micro) >= 0.85**.

**Note:** our starter `question_bank.csv` has 70 questions total (target is 150-300 per subject — see datasets/SOURCES.md). Expect this run to be a pipeline smoke test more than a metric-hitting run until the bank is scaled up with real past-paper questions.

## 0. Shared setup — mount Drive, confirm GPU, install deps
Run these three cells first in every notebook (per Phase 4 §0).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
# Expect to see "Tesla T4" — if this errors or shows no GPU, go to
# Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, then re-run.

In [ ]:
import os
DATA = '/content/drive/MyDrive/mentora_data'
MODELS = '/content/drive/MyDrive/mentora_models'
for m in ['model1_gap_classifier', 'model2_question_generator',
          'model3_trajectory_predictor', 'model4_career_matcher', 'model5_cv_ner']:
    os.makedirs(f'{MODELS}/{m}', exist_ok=True)

# NOTE: upload the repo's datasets/ folder to Google Drive at this path first
# (mentora_data/{raw,processed,labeled}/...) — see datasets/README.md.

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121  # Colab GPU wheel — do NOT reuse the CPU wheel from the backend's laptop plan
!pip install -q transformers peft accelerate sentence-transformers datasets evaluate seqeval spacy sacrebleu rouge_score

### Optional — Weights & Biases logging
Nice for live loss curves / a thesis screenshot. Skip entirely if you'd rather keep things simple — nothing below strictly needs it.

In [ ]:
# !pip install -q wandb
# import wandb
# wandb.login()  # paste your free-tier API key when prompted, once per session

## 1. Load and tokenize

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import pandas as pd, numpy as np

df = pd.read_csv(f'{DATA}/processed/m1/question_bank.csv')
topics = sorted(df['topic'].unique())
NUM_TOPICS = len(topics)
topic_to_idx = {t: i for i, t in enumerate(topics)}

def to_multihot(topic_str):
    vec = [0.0] * NUM_TOPICS
    for t in str(topic_str).split('|'):  # our data is single-topic; this also handles multi-topic if added later
        vec[topic_to_idx[t]] = 1.0
    return vec

df['labels'] = df['topic'].apply(to_multihot)

tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
def tokenize(batch):
    enc = tokenizer(batch['question_text'], truncation=True, padding='max_length', max_length=128)
    enc['labels'] = batch['labels']
    return enc

ds = Dataset.from_pandas(df[['question_text', 'labels']])
ds = ds.train_test_split(test_size=0.15, seed=42)
ds = ds.map(tokenize, batched=True)

## 2. Model + metrics

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'microsoft/deberta-v3-base', num_labels=NUM_TOPICS, problem_type='multi_label_classification'
)

from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    return {'f1_micro': f1_score(labels, preds, average='micro', zero_division=0)}

## 2b. Class-imbalance weighting — needed for a small, single-topic-per-question dataset

With ~70 questions spread across ~16-23 topics, each row's multi-hot label vector is almost all zeros (one positive topic out of many). Plain `BCEWithLogitsLoss` will happily learn to predict all-negative on a dataset this imbalanced/small, giving `f1_micro=0.0` even though `eval_loss` looks reasonable — that's a real failure mode, not a bug, and it's exactly what an initial run of this notebook hit. Weighting the loss toward positive examples (`pos_weight` in `BCEWithLogitsLoss`) is the standard fix for this. **This class is used two cells down, replacing the plain `Trainer`** — an earlier version of this notebook defined it here but never actually switched to it, which silently made this whole cell a no-op. Fixed now.

In [ ]:
import torch
import torch.nn as nn

labels_matrix = np.array(df['labels'].tolist())
num_pos = labels_matrix.sum(axis=0)
num_neg = len(labels_matrix) - num_pos
pos_weight = torch.tensor(np.where(num_pos > 0, num_neg / np.maximum(num_pos, 1), 1.0), dtype=torch.float32)
print('pos_weight per topic (higher = rarer topic, weighted more):')
print(dict(zip(topics, pos_weight.tolist())))

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(logits.device))
        loss = loss_fct(logits, labels.float())
        return (loss, outputs) if return_outputs else loss

## 3. Train — checkpoints every epoch to Drive, resumable

**Note 1:** `fp16` is off below. DeBERTa-v3's disentangled attention layer has a known incompatibility with PyTorch's `GradScaler` that throws `ValueError: Attempting to unscale FP16 gradients` partway through the first step — this isn't environment-specific, it's a model architecture issue. DeBERTa-base is small enough that fp32 training is still fine on a T4.

**Note 2:** `warmup_ratio` and `weight_decay` are set below because a flat `learning_rate=2e-5` for 30 epochs (240 total steps) with no warmup on a tiny dataset is a realistic way to destabilize training — if you previously saw the model's predicted probability come out nearly identical for every example (something like 0.3157 across the board) regardless of input text, that's this: a collapsed model, not an underconfident one. Warmup + weight decay is the standard fix.

**Note 3:** if you've run this cell before with different settings (e.g. without `WeightedTrainer` or without warmup), delete the old checkpoints first — `resume_from_checkpoint` will otherwise pick up optimizer/scheduler state that doesn't match the new loss function or schedule:
```python
# import shutil; shutil.rmtree(f'{MODELS}/model1_gap_classifier', ignore_errors=True)
```

In [ ]:
import glob

OUT_DIR = f'{MODELS}/model1_gap_classifier'

args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=30,  # bumped from 5 -- with only ~59 training rows this trains
                          # in under a minute on a T4, so more epochs is nearly free
    learning_rate=2e-5,
    warmup_ratio=0.1,     # ~24 warmup steps out of 240 -- prevents the early-training
                          # LR spike that a flat schedule can cause on this few steps
    weight_decay=0.01,
    save_strategy='epoch',
    eval_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_micro',
    fp16=False,  # see note above -- DeBERTa-v3 + fp16 GradScaler incompatibility
    report_to='wandb' if 'wandb' in dir() else 'none',
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=ds['train'], eval_dataset=ds['test'],
    compute_metrics=compute_metrics,
)

existing_checkpoints = glob.glob(f'{OUT_DIR}/checkpoint-*')
resume = sorted(existing_checkpoints)[-1] if existing_checkpoints else None
trainer.train(resume_from_checkpoint=resume)

## 4. Save final model

In [ ]:
trainer.save_model(f'{OUT_DIR}/final')
tokenizer.save_pretrained(f'{OUT_DIR}/final')
print(trainer.evaluate())

## 5. Threshold sweep — run this if F1 stayed at 0.0 while validation loss kept falling

That exact pattern (loss down, F1 stuck at exactly 0.0 every epoch) means the model IS learning, but with 16 label classes and ~1 correct label per example, BCE loss rewards predicting low probability almost everywhere long before it pushes the one correct label above the default 0.5 threshold — especially with this little training data. Check where the model's predicted probabilities actually sit before concluding it hasn't learned anything:

In [ ]:
import torch
from sklearn.metrics import f1_score

preds_output = trainer.predict(ds['test'])
logits = preds_output.predictions
probs = 1 / (1 + np.exp(-logits))
labels = preds_output.label_ids

print('Max predicted probability per example (should be well above 0.5 for a confident model):')
print(probs.max(axis=1))

for threshold in [0.5, 0.4, 0.3, 0.2, 0.1]:
    preds = (probs >= threshold).astype(int)
    f1 = f1_score(labels, preds, average='micro', zero_division=0)
    print(f'threshold={threshold}: f1_micro={f1:.3f}')

**Reading the result:**
- **Check the max-probability line first.** If it's nearly identical across every example (e.g. all ~0.31, varying only in the 3rd decimal place), the model has collapsed to predicting a near-constant output regardless of input text — it hasn't learned anything, and no threshold will fix that. This is what the warmup/weight-decay/class-weighting fixes above (§2b/§3) address.
- If probabilities vary meaningfully example-to-example and a lower threshold (e.g. 0.2-0.3) recovers a reasonable F1, the model learned the right ranking, it's just underconfident from too little data — usable with that lower threshold in the app's inference code, though growing the question bank is still the real fix long-term.
- If probabilities vary but F1 stays near 0 even at threshold 0.1, the 70-question starter set is genuinely too small (59 training rows / 16 classes averages under 4 examples per class) — scaling up `datasets/scripts/generate_m1_starter_bank.py` with real past-paper questions per `datasets/SOURCES.md` is the fix, not more epochs or a different threshold.

## Target metric: F1 (micro) >= 0.85

If it plateaus lower:
- Sweep the classification threshold (0.3/0.4/0.5/0.6) on validation
- Check per-topic example counts — merge near-unlearnable rare subtopics into their parent
- Highest-leverage fix: grow the M1 question bank past the 70-question starter set (also directly benefits M2)